##L11: Auto Loader: incremental file ingestion

###Goal
Ingest new monthly files automatically and prove only new files are processed on each run.

###Step 1. Move one month out of the landing folder
This lets you simulate a new file arriving. In a notebook:

In [0]:
base = "/Volumes/da/bronze/landing"
dbutils.fs.mkdirs(f"{base}/hold")
dbutils.fs.mv(f"{base}/yellow_trips/yellow_tripdata_2025-02.parquet", f"{base}/hold/yellow_tripdata_2025-02.parquet")
print([f.name for f in dbutils.fs.ls(f"{base}/yellow_trips")])

###Step 2. Run Auto Loader on what remains
schemaLocation stores the inferred schema; the checkpoint remembers which files are done.

In [0]:
from pyspark.sql import functions as F

base = "/Volumes/da/bronze/landing"
meta = f"{base}/_meta/yellow_al"

def run_auto_loader():
    q = (spark.readStream.format("cloudFiles")
         .option("cloudFiles.format", "parquet")
         .option("cloudFiles.schemaLocation", f"{meta}/schema")
         .load(f"{base}/yellow_trips")
         .select("*", F.col("_metadata.file_name").alias("source_file"),
                      F.current_timestamp().alias("ingested_at"))
         .writeStream
         .option("checkpointLocation", f"{meta}/checkpoint")
         .option("mergeSchema", "true")
         .trigger(availableNow=True)
         .toTable("da.bronze.yellow_trips_al"))
    q.awaitTermination()

run_auto_loader()

In [0]:
%sql
SELECT source_file, COUNT(*) FROM da.bronze.yellow_trips_al GROUP BY source_file;

###Step 3. Add the "new" file and run again

In [0]:
dbutils.fs.mv(f"{base}/hold/yellow_tripdata_2025-02.parquet", f"{base}/yellow_trips/yellow_tripdata_2025-02.parquet")
run_auto_loader()

In [0]:
%sql
SELECT source_file, COUNT(*) FROM da.bronze.yellow_trips_al GROUP BY source_file ORDER BY 1;
DESCRIBE HISTORY da.bronze.yellow_trips_al;

Only the February file should have been added. Open the history: the latest streaming update shows numOutputRows equal to the February row count.

###Step 4. Run it a third time
Run run_auto_loader() again with no new files. It should finish quickly and add nothing. This is idempotent, incremental ingestion.

In [0]:
run_auto_loader()

###Step 5. Observe schema evolution (optional but valuable)
Reset: drop the table, delete the _meta folder, move January and February out, run (December only), move January back, run again.

In [0]:
spark.sql("DROP TABLE IF EXISTS da.bronze.yellow_trips_al")
dbutils.fs.rm(f"{base}/_meta/yellow_al", True)
for m in ["2025-01", "2025-02"]:
    dbutils.fs.mv(f"{base}/yellow_trips/yellow_tripdata_{m}.parquet", f"{base}/hold/yellow_tripdata_{m}.parquet")
run_auto_loader()
dbutils.fs.mv(f"{base}/hold/yellow_tripdata_2025-01.parquet", f"{base}/yellow_trips/yellow_tripdata_2025-01.parquet")
run_auto_loader()